## 0. 공통 Import

# 02. IP Usage Analysis

> **목표**: *DB가 없으면 grep으로 수 시간 걸릴 질문을 1초에 답하는 효과* 시연

YAML 파일을 직접 열어 확인하던 작업들:
- "어느 IP가 어떤 variant에 쓰이나?" → `jsonb_object_keys` 1 쿼리
- "ISP TNR vs MFC 전력 분포?" → `jsonb_array_elements` unnest
- "해상도별로 어떤 IP 조합이 필요한가?" → cross join + pivot
- "LLC 경쟁 구조" → `jsonb_each_text` 1 쿼리

**Sections**
1. IP별 사용 빈도
2. ISP Submodule 전력 분해 (Before/After)
3. IP × Resolution 매트릭스
4. Throughput 요구사항 분포
5. LLC 자원 경쟁 (Demo Story)

---

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

from demo.notebooks.utils.db_connection import get_engine, query_df
from demo.notebooks.utils.plot_theme import (
    apply_theme, FEASIBILITY_COLORS, IP_CATEGORY_COLORS,
    SW_VERSION_PALETTE, ACCENT, BG_LIGHT, plotly_layout,
)

apply_theme()
pd.set_option('display.max_colwidth', 80)
ENGINE = get_engine()
print('✓ ready')

✓ ready


---
## Section 1. IP별 사용 빈도

**질문**: *어떤 IP instance가 몇 개 variant / scenario에서 요구되는가?*

- `ip_requirements` JSONB의 키(`isp0`, `mfc`, `llc` …)를 `jsonb_object_keys`로 전개
- Pipeline nodes와 JOIN해 abstract ip_ref(예: `ip-isp-v12`)까지 매핑

In [2]:
df_ip_freq = query_df("""
    WITH ip_instances AS (
        -- variant마다 ip_requirements 키를 행으로 전개
        SELECT
            sv.scenario_id,
            sv.id                                   AS variant_id,
            sv.severity,
            sv.design_conditions->>'resolution'     AS resolution,
            ip_key                                  AS ip_instance
        FROM scenario_variants sv,
             jsonb_object_keys(sv.ip_requirements) AS ip_key
        WHERE sv.ip_requirements <> '{}'::jsonb
    ),
    node_map AS (
        -- pipeline node id → ip_ref 매핑
        SELECT
            s.id                        AS scenario_id,
            node->>'id'                 AS instance_id,
            node->>'ip_ref'             AS ip_ref
        FROM scenarios s,
             jsonb_array_elements(s.pipeline->'nodes') AS node
    )
    SELECT
        ii.ip_instance,
        COALESCE(nm.ip_ref, '(unmapped)')  AS ip_ref,
        COUNT(DISTINCT ii.variant_id)      AS variant_count,
        COUNT(DISTINCT ii.scenario_id)     AS scenario_count,
        string_agg(DISTINCT ii.resolution, ', '
                   ORDER BY ii.resolution) AS resolutions,
        string_agg(DISTINCT ii.severity,   ', '
                   ORDER BY ii.severity)   AS severities
    FROM ip_instances ii
    LEFT JOIN node_map nm
           ON nm.scenario_id  = ii.scenario_id
          AND nm.instance_id  = ii.ip_instance
    GROUP BY ii.ip_instance, nm.ip_ref
    ORDER BY variant_count DESC, ip_instance
""", ENGINE)

display(HTML('<h4>IP Instance별 사용 빈도</h4>'))
display(df_ip_freq)

,ip_instance,ip_ref,variant_count,scenario_count,resolutions,severities
0,isp0,ip-isp-v12,2,1,"8K, UHD","critical, heavy"
1,mfc,ip-mfc-v14,2,1,"8K, UHD","critical, heavy"
2,llc,ip-llc-v2,1,1,UHD,heavy


In [ ]:
fig1, ax1 = plt.subplots(figsize=(9, 4))

# ip_ref 기반 카테고리 색상
def _ip_color(ip_ref: str) -> str:
    for cat, col in IP_CATEGORY_COLORS.items():
        if cat in ip_ref:
            return col
    return IP_CATEGORY_COLORS['other']

colors = [_ip_color(r) for r in df_ip_freq['ip_ref']]
bars = ax1.barh(df_ip_freq['ip_instance'], df_ip_freq['variant_count'],
                color=colors, edgecolor='white', linewidth=0.8)
ax1.bar_label(bars, fmt='%d variant', padding=4, fontsize=9, color=ACCENT)
ax1.set_xlabel('Variant 수')
ax1.set_title('IP Instance별 사용 빈도', fontsize=13, fontweight='bold', color=ACCENT)
ax1.invert_yaxis()

# 범례
legend_patches = [
    mpatches.Patch(color=col, label=cat)
    for cat, col in IP_CATEGORY_COLORS.items()
    if col in colors
]
ax1.legend(handles=legend_patches, title='IP Category',
           loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('s1_ip_freq.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s1_ip_freq.png')

**해석**
- `isp0`와 `mfc`는 모든 실 variant에서 요구됨 → 카메라 파이프라인의 필수 IP
- `llc`는 UHD 이상 고해상도 variant에서만 명시적 할당 요구
- `CSIS`, `DPU`는 pipeline에는 존재하지만 ip_requirements에 명시 안 됨 →
  현재 Throughput 제약이 없음을 의미 (추후 추가 필요)

> YAML grep이었다면: `grep -r 'required_throughput' ./*.yaml` 로 찾더라도
> IP instance ↔ scenario 매핑은 수작업이 필요했을 것.

---
## Section 2. ISP Submodule 전력 분해 (Before / After)

**질문**: *LLC 버그 수정 전후로 ISP 내부 submodule 전력이 어떻게 변하는가?*

- `evidence.ip_breakdown` JSONB 배열 → `jsonb_array_elements`로 IP별 전개
- 각 IP의 `submodules` 배열을 추가로 unnest
- sw-vendor-v1.2.3 vs v1.3.0 비교

In [ ]:
df_submod = query_df("""
    WITH ip_level AS (
        SELECT
            e.id                              AS evidence_id,
            e.sw_version_hint,
            e.overall_feasibility,
            breakdown->>'ip'                  AS ip_ref,
            (breakdown->>'power_mW')::float   AS ip_power_mw,
            breakdown->'submodules'           AS submodules
        FROM evidence e,
             jsonb_array_elements(e.ip_breakdown) AS breakdown
        WHERE e.kind = 'evidence.simulation'
          AND e.ip_breakdown IS NOT NULL
    )
    SELECT
        il.sw_version_hint,
        il.overall_feasibility,
        il.ip_ref,
        il.ip_power_mw,
        sub->>'sub'              AS submodule,
        (sub->>'power_mW')::float AS sub_power_mw
    FROM ip_level il,
         jsonb_array_elements(il.submodules) AS sub
    WHERE jsonb_array_length(il.submodules) > 0
    ORDER BY il.sw_version_hint, il.ip_ref, sub_power_mw DESC
""", ENGINE)

display(HTML('<h4>IP Submodule 전력 분해 (simulation evidence)</h4>'))
display(df_submod)

In [ ]:
# ISP submodule 전력 — Before/After grouped bar
df_isp = df_submod[df_submod['ip_ref'] == 'ip-isp-v12'].copy()

fig2, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

sw_versions = df_isp['sw_version_hint'].unique()
palette = {v: SW_VERSION_PALETTE[i] for i, v in enumerate(sorted(sw_versions))}

for ax, sw_ver in zip(axes, sorted(sw_versions)):
    subset = df_isp[df_isp['sw_version_hint'] == sw_ver]
    feasibility = subset['overall_feasibility'].iloc[0]
    fc = FEASIBILITY_COLORS.get(feasibility, '#95A5A6')

    bars = ax.bar(subset['submodule'], subset['sub_power_mw'],
                  color=palette[sw_ver], edgecolor='white', linewidth=0.8)
    ax.bar_label(bars, fmt='%.0f mW', padding=3, fontsize=9)
    ax.set_title(f'{sw_ver}\n({feasibility})',
                 fontsize=10, fontweight='bold',
                 color=fc)
    ax.set_ylabel('Power (mW)')
    ax.set_ylim(0, df_isp['sub_power_mw'].max() * 1.25)

    # IP 전체 전력 표시
    ip_total = subset['ip_power_mw'].iloc[0]
    ax.axhline(ip_total, color=fc, linestyle='--', linewidth=1.2, alpha=0.7)
    ax.text(0.98, ip_total + 5, f'IP total: {ip_total:.0f} mW',
            ha='right', va='bottom', fontsize=8, color=fc,
            transform=ax.get_yaxis_transform())

fig2.suptitle('ISP-v12 Submodule Power Breakdown\nBefore vs After LLC Fix',
              fontsize=13, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.savefig('s2_isp_submodule_power.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s2_isp_submodule_power.png')

In [ ]:
# 전체 IP 전력 합계 비교 (stacked bar — sw 버전별)
df_ip_total = query_df("""
    SELECT
        e.sw_version_hint,
        e.overall_feasibility,
        breakdown->>'ip'                AS ip_ref,
        (breakdown->>'power_mW')::float AS power_mw
    FROM evidence e,
         jsonb_array_elements(e.ip_breakdown) AS breakdown
    WHERE e.kind = 'evidence.simulation'
    ORDER BY e.sw_version_hint, ip_ref
""", ENGINE)

pivot = df_ip_total.pivot(index='sw_version_hint', columns='ip_ref', values='power_mw').fillna(0)
feasibility_map = df_ip_total.groupby('sw_version_hint')['overall_feasibility'].first()

fig3, ax3 = plt.subplots(figsize=(8, 4))
ip_colors = [_ip_color(c) for c in pivot.columns]
pivot.plot(kind='bar', stacked=True, ax=ax3, color=ip_colors,
           edgecolor='white', linewidth=0.6)

ax3.set_title('IP별 전력 합계 — sw 버전 비교', fontsize=12, fontweight='bold', color=ACCENT)
ax3.set_ylabel('Power (mW)')
ax3.set_xlabel('')
ax3.tick_params(axis='x', rotation=15)

# feasibility 레이블
for i, (sw, row) in enumerate(pivot.iterrows()):
    feas = feasibility_map[sw]
    fc = FEASIBILITY_COLORS.get(feas, '#95A5A6')
    ax3.text(i, row.sum() + 20, feas, ha='center', va='bottom',
             fontsize=8, color=fc, fontweight='bold')

ax3.legend(title='IP', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('s2_ip_power_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s2_ip_power_stacked.png')

**해석**
- sw-v1.2.3: ISP.TNR 420 mW → sw-v1.3.0: 320 mW (**-24%**)
  LLC thrashing으로 TNR이 메모리를 재요청하며 낭비하던 전력이 제거됨
- MFC도 520 → 450 mW (**-13%**): LLC 정상화로 버퍼 재할당 오버헤드 감소
- 전체 전력 2350 → 2150 mW (**-200 mW, -8.5%**)

> 이 분석을 YAML grep으로 하려면:
> evidence YAML 2개를 열어 ip_breakdown 수동 비교 → 오류 가능성 높음

---
## Section 3. IP × Resolution 매트릭스

**질문**: *해상도별로 어떤 IP 조합이 필요한가?*

- Pipeline nodes(전체 IP 집합) × variant design_conditions.resolution 교차
- ip_requirements에 명시된 IP는 'required', pipeline에만 있으면 'pipeline_only'

In [ ]:
df_matrix = query_df("""
    WITH pipeline_ips AS (
        -- 시나리오 파이프라인의 전체 IP 목록
        SELECT
            s.id               AS scenario_id,
            node->>'id'        AS instance_id,
            node->>'ip_ref'    AS ip_ref
        FROM scenarios s,
             jsonb_array_elements(s.pipeline->'nodes') AS node
    ),
    variant_resolution AS (
        SELECT
            sv.scenario_id,
            sv.id                                    AS variant_id,
            COALESCE(sv.design_conditions->>'resolution',
                     '(derived)')                    AS resolution,
            sv.severity
        FROM scenario_variants sv
    ),
    variant_req_keys AS (
        -- variant에서 명시적으로 요구된 IP instance 집합
        SELECT
            sv.id  AS variant_id,
            ip_key AS ip_instance
        FROM scenario_variants sv,
             jsonb_object_keys(sv.ip_requirements) AS ip_key
        WHERE sv.ip_requirements <> '{}'::jsonb
    )
    SELECT
        vr.resolution,
        vr.severity,
        vr.variant_id,
        pi.instance_id,
        pi.ip_ref,
        CASE
            WHEN vrk.ip_instance IS NOT NULL THEN 'required'
            ELSE 'pipeline_only'
        END AS usage_type
    FROM variant_resolution vr
    JOIN pipeline_ips pi ON pi.scenario_id = vr.scenario_id
    LEFT JOIN variant_req_keys vrk
           ON vrk.variant_id  = vr.variant_id
          AND vrk.ip_instance = pi.instance_id
    ORDER BY vr.resolution, pi.instance_id
""", ENGINE)

display(HTML('<h4>IP × Resolution 교차 (전체)</h4>'))
display(df_matrix)

In [ ]:
# Pivot: resolution × instance_id → usage_type을 수치로
usage_map = {'required': 2, 'pipeline_only': 1}
df_heat = df_matrix.copy()
df_heat['usage_num'] = df_heat['usage_type'].map(usage_map)

pivot_heat = df_heat.pivot_table(
    index='instance_id', columns='resolution',
    values='usage_num', aggfunc='max'
).fillna(0)

# 컬럼 순서 정리
col_order = [c for c in ['FHD','UHD','8K','(derived)'] if c in pivot_heat.columns]
pivot_heat = pivot_heat[col_order]

fig4, ax4 = plt.subplots(figsize=(7, 4))
cmap = plt.cm.colors.ListedColormap([BG_LIGHT, '#AED6F1', '#2980B9'])

im = ax4.imshow(pivot_heat.values, cmap='Blues', vmin=0, vmax=2, aspect='auto')

# 레이블
label_map = {0: '—', 1: 'pipeline\nonly', 2: 'required'}
for i in range(pivot_heat.shape[0]):
    for j in range(pivot_heat.shape[1]):
        val = int(pivot_heat.values[i, j])
        color = 'white' if val == 2 else ACCENT
        ax4.text(j, i, label_map[val], ha='center', va='center',
                 fontsize=9, color=color, fontweight='bold' if val == 2 else 'normal')

ax4.set_xticks(range(len(pivot_heat.columns)))
ax4.set_xticklabels(pivot_heat.columns, fontsize=10)
ax4.set_yticks(range(len(pivot_heat.index)))
ax4.set_yticklabels(pivot_heat.index, fontsize=10)
ax4.set_title('IP × Resolution 사용 매트릭스', fontsize=12, fontweight='bold', color=ACCENT)
ax4.set_xlabel('Resolution', fontsize=10)
ax4.set_ylabel('IP Instance', fontsize=10)

# 범례
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2980B9', label='required (ip_requirements 명시)'),
    Patch(facecolor='#AED6F1', label='pipeline only (연결은 있으나 요구사항 미명시)'),
    Patch(facecolor=BG_LIGHT,  edgecolor='#CCC', label='미사용'),
]
ax4.legend(handles=legend_elements, loc='upper right',
           bbox_to_anchor=(1.0, -0.15), fontsize=8, ncol=1)

plt.tight_layout()
plt.savefig('s3_ip_resolution_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s3_ip_resolution_matrix.png')

**해석**
- `isp0`, `mfc`는 UHD·8K 모두에서 **required** — 핵심 병목 IP
- `llc`는 UHD에서만 required, 8K에서는 미명시 → **8K variant에 LLC 요구사항 추가 필요**
- `csis0`, `dpu`는 pipeline에 존재하지만 ip_requirements 미작성 → 검토 action item

> 이 매트릭스를 YAML grep으로: 5개 IP × 3개 해상도 조합을 수동으로 파일 교차 확인.

---
## Section 4. Throughput 요구사항 분포

**질문**: *IP 처리량 요구사항이 resolution·severity별로 얼마나 차이 나는가?*

- `ip_requirements.isp0.required_throughput_mpps` 추출
- Design axes의 capability max(800 Mpps)와 비교

In [ ]:
df_tp = query_df("""
    SELECT
        sv.id                                                        AS variant_id,
        sv.severity,
        COALESCE(sv.design_conditions->>'resolution', '(derived)')   AS resolution,
        COALESCE(sv.design_conditions->>'fps', '?')                  AS fps,
        COALESCE(sv.design_conditions->>'codec',  '?')               AS codec,
        -- ISP throughput
        (sv.ip_requirements->'isp0'->>'required_throughput_mpps'
        )::float                                                     AS isp_throughput_mpps,
        -- MFC codec 요구사항
        sv.ip_requirements->'mfc'->>'required_codec'                 AS mfc_codec,
        sv.ip_requirements->'mfc'->>'required_level'                 AS mfc_level
    FROM scenario_variants sv
    WHERE sv.ip_requirements->'isp0'->>'required_throughput_mpps' IS NOT NULL
    ORDER BY isp_throughput_mpps DESC
""", ENGINE)

display(HTML('<h4>Throughput 요구사항 (ISP)</h4>'))
display(df_tp)

In [ ]:
ISP_CAPABILITY_MAX = 800.0  # ip-isp-v12 high_throughput mode

res_colors = {'UHD': '#3498DB', '8K': '#E74C3C', 'FHD': '#2ECC71', '(derived)': '#95A5A6'}
df_tp['color'] = df_tp['resolution'].map(res_colors)

fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(12, 4))

# --- 좌: 가로 막대 (capability max 대비) ---
bars5 = ax5a.barh(
    df_tp['variant_id'], df_tp['isp_throughput_mpps'],
    color=df_tp['color'], edgecolor='white'
)
ax5a.axvline(ISP_CAPABILITY_MAX, color='#E74C3C', linestyle='--',
             linewidth=1.5, label=f'Capability max ({ISP_CAPABILITY_MAX:.0f} Mpps)')
ax5a.bar_label(bars5, fmt='%.0f Mpps', padding=4, fontsize=9)
ax5a.set_xlabel('Required Throughput (Mpps)')
ax5a.set_title('ISP Throughput 요구사항\nvs Capability Max', fontsize=11, fontweight='bold', color=ACCENT)
ax5a.set_xlim(0, ISP_CAPABILITY_MAX * 1.3)
ax5a.invert_yaxis()
ax5a.legend(fontsize=8)

# --- 우: headroom 비율 (capability 대비 여유) ---
df_tp['headroom_pct'] = (ISP_CAPABILITY_MAX - df_tp['isp_throughput_mpps']) / ISP_CAPABILITY_MAX * 100
hbar = ax5b.barh(
    df_tp['variant_id'], df_tp['headroom_pct'],
    color=['#27AE60' if h > 20 else '#E67E22' if h > 0 else '#C0392B'
           for h in df_tp['headroom_pct']],
    edgecolor='white'
)
ax5b.bar_label(hbar, fmt='%.1f%%', padding=4, fontsize=9)
ax5b.axvline(0, color='#E74C3C', linewidth=1)
ax5b.set_xlabel('Capability Headroom (%)')
ax5b.set_title('ISP Headroom\n(음수 = 초과, 불가능)', fontsize=11, fontweight='bold', color=ACCENT)
ax5b.invert_yaxis()

plt.suptitle('ISP-v12 Throughput 요구사항 분석', fontsize=13, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.savefig('s4_throughput_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s4_throughput_analysis.png')

**해석**
- **UHD60**: 498 Mpps 요구 → capability 800 Mpps 대비 **37.8% 여유** — production feasible
- **8K120**: 3,981 Mpps 요구 → capability 800 Mpps 대비 **-397% 초과** → WARN_AND_CAP 정책 적용, exploration_only
- 8K120는 현재 ISP 세대(v12)로는 실리콘 지원 불가 — 다음 IP 세대(v13+) 또는 분산 처리 필요

> 이 계산을 수동으로: ISP YAML → `capabilities.operating_modes` 확인 → variant YAML → 수치 비교.
> DB 쿼리: **1초**. 수동 grep: **10분+**

---
## Section 5. LLC 자원 경쟁 (Demo Story)

**질문**: *각 variant에서 LLC를 어느 IP가 얼마나 요구하는가? 왜 LLC thrashing이 발생했는가?*

- `ip_requirements.llc.required_allocations` JSONB → `jsonb_each_text`로 consumer별 행 전개
- variant별 LLC 총 요구량 vs 실제 할당 정책 비교
- iss-LLC-thrashing-0221 이슈와 연결

In [ ]:
df_llc = query_df("""
    SELECT
        sv.id                                                AS variant_id,
        COALESCE(sv.design_conditions->>'resolution',
                 '(derived)')                               AS resolution,
        sv.severity,
        alloc.key                                           AS consumer,
        alloc.value                                         AS allocation_str,
        -- MB 수치 추출
        REPLACE(alloc.value, 'MB', '')::float               AS allocation_mb
    FROM scenario_variants sv,
         jsonb_each_text(
             sv.ip_requirements->'llc'->'required_allocations'
         ) AS alloc
    WHERE sv.ip_requirements->'llc'->'required_allocations' IS NOT NULL
      AND sv.ip_requirements->'llc'->'required_allocations' <> '{}'::jsonb
    ORDER BY sv.id, allocation_mb DESC
""", ENGINE)

# 이슈 정보 조회
df_issue = query_df("""
    SELECT
        id,
        metadata->>'title'             AS title,
        metadata->>'status'            AS status,
        resolution->>'fix_sw_ref'      AS fixed_in_sw,
        resolution->>'fix_description' AS fix_description
    FROM issues
    WHERE id = 'iss-LLC-thrashing-0221'
""", ENGINE)

display(HTML('<h4>LLC required_allocations 전개</h4>'))
display(df_llc)
display(HTML('<h4>관련 이슈</h4>'))
display(df_issue)

In [ ]:
# Stacked bar: variant × consumer
pivot_llc = df_llc.pivot(index='variant_id', columns='consumer', values='allocation_mb').fillna(0)

consumer_colors = {'ISP.TNR': '#3498DB', 'MFC': '#E67E22', 'ISP.3AA0': '#9B59B6'}
bar_colors = [consumer_colors.get(c, '#95A5A6') for c in pivot_llc.columns]

fig6, ax6 = plt.subplots(figsize=(8, 4.5))
pivot_llc.plot(kind='bar', stacked=True, ax=ax6,
               color=bar_colors, edgecolor='white', linewidth=0.8)

# sw-v1.2.3에서의 실제 할당 vs 요구량 — 이슈 포인트 시각화
# sw-v1.2.3: LLC shared mode → per-consumer 구분 안 됨 (버그 원인)
TNR_ACTUAL_SW123  = 2.0   # 요구량 그대로지만 shared pool에서 경쟁
LLC_TOTAL_SW123   = 4.0   # 실제 총 LLC 용량 (가정)
ax6.axhline(LLC_TOTAL_SW123, color='#E74C3C', linestyle='--', linewidth=1.5,
            label=f'LLC 총 용량 (sw-v1.2.3 shared: {LLC_TOTAL_SW123:.0f} MB)')

# 총 요구량 레이블
for i, (variant, row) in enumerate(pivot_llc.iterrows()):
    total = row.sum()
    ax6.text(i, total + 0.05, f'{total:.0f} MB total',
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=ACCENT)

ax6.set_title('Variant별 LLC 자원 요구량 (required_allocations)',
              fontsize=12, fontweight='bold', color=ACCENT)
ax6.set_ylabel('LLC Allocation (MB)')
ax6.set_xlabel('')
ax6.tick_params(axis='x', rotation=15)
ax6.legend(title='Consumer', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

# 어노테이션 — 이슈 포인트
ax6.annotate(
    'sw-v1.2.3: LLC shared mode\n→ ISP.TNR와 MFC가 동일 풀 경쟁\n→ TNR 부족 → thrashing 발생',
    xy=(0, 2.0), xytext=(0.55, 3.5),
    fontsize=8, color='#C0392B',
    arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.5),
)

plt.tight_layout()
plt.savefig('s5_llc_contention.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s5_llc_contention.png')

In [ ]:
# LLC 이슈 → Waiver → Review 흐름 (Plotly timeline)
df_timeline = pd.DataFrame([
    dict(Task='iss-LLC-thrashing-0221', Start='2026-02-21', Finish='2026-03-05',
         Type='Issue', Detail='LLC thrashing 발견 (sw-v1.2.1 regression)'),
    dict(Task='waiver-LLC-thrashing', Start='2026-04-17', Finish='2026-09-30',
         Type='Waiver', Detail='A0 + sw-v1.2.x 한정 waiver (MFA 승인)'),
    dict(Task='rev-sw123 (WARN)', Start='2026-04-19', Finish='2026-04-19',
         Type='Review WARN', Detail='gate=WARN, approved_with_waiver'),
    dict(Task='sw-vendor-v1.3.0 출시', Start='2026-04-01', Finish='2026-04-01',
         Type='SW Release', Detail='LLC_per_ip_partition: enabled'),
    dict(Task='rev-sw130 (PASS)', Start='2026-04-19', Finish='2026-04-19',
         Type='Review PASS', Detail='gate=PASS, approved (waiver 불필요)'),
])

color_map = {
    'Issue':       '#E74C3C',
    'Waiver':      '#E67E22',
    'Review WARN': '#F39C12',
    'SW Release':  '#27AE60',
    'Review PASS': '#2ECC71',
}

fig7 = px.timeline(
    df_timeline, x_start='Start', x_end='Finish',
    y='Task', color='Type',
    text='Detail',
    color_discrete_map=color_map,
    title='LLC Thrashing 이슈 → 해결 타임라인',
)
fig7.update_traces(textposition='inside', insidetextanchor='start')
fig7.update_layout(
    **plotly_layout('LLC Thrashing: Issue → Waiver → Fix → PASS Review'),
    xaxis_title='Date',
    showlegend=True,
    height=350,
)
fig7.show()

**해석 — LLC 이슈의 원인이 여기서 보인다**

| 항목 | 값 |
|------|----|
| ISP.TNR 요구 할당 | **2 MB** |
| MFC 요구 할당 | **1 MB** |
| 합계 | **3 MB** |
| sw-v1.2.3 LLC 모드 | `shared` (per-IP 구분 없음) |
| 실제 총 LLC | ~4 MB |

**문제**: sw-v1.2.3에서 LLC를 shared pool로 운영하면 MFC가 ISP.TNR의 할당 영역을 침범 가능.
열이 올라가는 조건(hot, UHD) → TNR이 캐시 미스 반복 → **thrashing**.

**수정**: sw-v1.3.0에서 `LLC_per_ip_partition: enabled` → ISP.TNR 2 MB 고정 파티션 보장
→ 경쟁 제거, violation 0건, `production_ready` 판정.

---

### Summary — "1초 분석"이 보여주는 것

| 질문 | DB 쿼리 시간 | YAML grep 시간 |
|------|-------------|----------------|
| 어느 IP가 몇 개 variant에 쓰이나? | < 0.1s | ~10분 (수동 집계) |
| 해상도별 IP 조합 매트릭스 | < 0.1s | ~30분 (교차 대조) |
| ISP submodule 전력 Before/After | < 0.1s | ~20분 (2개 YAML 비교) |
| LLC 경쟁 consumer 분해 | < 0.1s | ~15분 (여러 YAML 추적) |
| 이슈→waiver→review 타임라인 | < 0.1s | 불가 (파일 간 추적 불가) |